In [ ]:
!pip install unsloth trl peft transformers datasets wandb python-dotenv -q
import os
os.environ["WANDB_PROJECT"] = "tweet-scorer-finetuning"
# Set these as Kaggle secrets before running:
# WANDB_API_KEY, HF_TOKEN

In [ ]:
from datasets import load_dataset
dataset = load_dataset("sudar/tweet-scorer-dataset")
print(dataset)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
import wandb
from trl import SFTTrainer, SFTConfig

wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(project="tweet-scorer-finetuning", name="llama3-8b-qlora-r16")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset["train"], eval_dataset=dataset["validation"],
    args=SFTConfig(
        output_dir="/kaggle/working/tweet-scorer",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=True, bf16=False,
        logging_steps=10,
        eval_strategy="steps", eval_steps=50,
        save_strategy="steps", save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="wandb",
        dataset_text_field="messages",
        max_seq_length=2048,
        packing=True,
        dataset_num_proc=2,
    ),
)
trainer.train()
wandb.finish()

In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
model.push_to_hub("sudar/tweet-scorer-llama3-8b")
tokenizer.push_to_hub("sudar/tweet-scorer-llama3-8b")
print("Adapter pushed to HF Hub.")